# MarianMT — Spanish ↔ English General Base Model
**iDISC Personalized Translation Engines · Phase 1: Baseline**

This notebook replaces the NLLB-200 general base model with **MarianMT** (`Helsinki-NLP/opus-mt-*`) — purpose-built bidirectional EN↔SP models that are lighter, faster, and already used in the domain-specific fine-tuning pipeline.

This is the **pre-fine-tuning baseline**. All future fine-tuned models will be measured against these scores.

---
**Models used:**
| Direction | Model | Size |
|---|---|---|
| ES→EN | `Helsinki-NLP/opus-mt-es-en` | ~300 MB |
| EN→ES | `Helsinki-NLP/opus-mt-en-es` | ~300 MB |

> Enable GPU accelerator in *Settings → Accelerator → GPU T4* before running.

In [1]:
%%capture
!pip install transformers sentencepiece sacrebleu unbabel-comet bert-score accelerate

### Configuration & Imports

In [2]:
import logging
import time
from dataclasses import dataclass, field
from typing import Optional

import torch
from transformers import MarianMTModel, MarianTokenizer

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
log = logging.getLogger(__name__)

EN_ES_MODEL = "Helsinki-NLP/opus-mt-en-es"
ES_EN_MODEL = "Helsinki-NLP/opus-mt-es-en"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU   : Tesla T4
VRAM  : 15.6 GB


### Data Classes

In [3]:
@dataclass
class TranslationResult:
    source_text: str
    translated_text: str
    source_lang: str
    target_lang: str
    latency_ms: float
    device: str
    model: str
    score: Optional[float] = None
    flagged_for_review: bool = False


@dataclass
class BatchTranslationResult:
    results: list = field(default_factory=list)
    total_latency_ms: float = 0.0
    avg_latency_ms: float = 0.0

### MarianTranslator Class

In [4]:
class MarianTranslator:
    """
    General Base Model wrapper for MarianMT (Spanish ↔ English).
    Uses separate Helsinki-NLP models per direction — no language codes needed.
    """

    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        log.info(f"Loading ES→EN model: {ES_EN_MODEL}")
        self.es_en_tokenizer = MarianTokenizer.from_pretrained(ES_EN_MODEL)
        self.es_en_model = MarianMTModel.from_pretrained(ES_EN_MODEL).to(self.device)
        self.es_en_model.eval()

        log.info(f"Loading EN→ES model: {EN_ES_MODEL}")
        self.en_es_tokenizer = MarianTokenizer.from_pretrained(EN_ES_MODEL)
        self.en_es_model = MarianMTModel.from_pretrained(EN_ES_MODEL).to(self.device)
        self.en_es_model.eval()

        log.info("Both models ready")

    def _get_model_and_tokenizer(self, src: str, tgt: str):
        if src == "es" and tgt == "en":
            return self.es_en_model, self.es_en_tokenizer, ES_EN_MODEL
        elif src == "en" and tgt == "es":
            return self.en_es_model, self.en_es_tokenizer, EN_ES_MODEL
        else:
            raise ValueError(f"Unsupported pair: {src}→{tgt}. Use 'es'/'en'.")

    def translate(
        self,
        text: str,
        src: str = "es",
        tgt: str = "en",
        num_beams: int = 4,
        max_length: int = 512,
    ) -> TranslationResult:
        model, tokenizer, model_name = self._get_model_and_tokenizer(src, tgt)

        inputs = tokenizer(
            text, return_tensors="pt", padding=True,
            truncation=True, max_length=max_length
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        t0 = time.perf_counter()
        with torch.no_grad():
            output_ids = model.generate(**inputs, num_beams=num_beams)
        latency_ms = (time.perf_counter() - t0) * 1000

        translated = tokenizer.decode(output_ids[0], skip_special_tokens=True)

        return TranslationResult(
            source_text=text,
            translated_text=translated,
            source_lang=src,
            target_lang=tgt,
            latency_ms=round(latency_ms, 2),
            device=self.device,
            model=model_name,
        )

    def translate_batch(
        self,
        texts: list,
        src: str = "es",
        tgt: str = "en",
        batch_size: int = 8,
        num_beams: int = 4,
        max_length: int = 512,
    ) -> BatchTranslationResult:
        model, tokenizer, model_name = self._get_model_and_tokenizer(src, tgt)
        all_results = []
        total_start = time.perf_counter()

        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            inputs = tokenizer(
                batch, return_tensors="pt", padding=True,
                truncation=True, max_length=max_length
            )
            inputs = {k: v.to(self.device) for k, v in inputs.items()}

            t0 = time.perf_counter()
            with torch.no_grad():
                output_ids = model.generate(**inputs, num_beams=num_beams)
            batch_latency = (time.perf_counter() - t0) * 1000
            per_seg = batch_latency / len(batch)

            decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
            for src_text, tgt_text in zip(batch, decoded):
                all_results.append(TranslationResult(
                    source_text=src_text,
                    translated_text=tgt_text,
                    source_lang=src,
                    target_lang=tgt,
                    latency_ms=round(per_seg, 2),
                    device=self.device,
                    model=model_name,
                ))
            log.info(f"Batch {i // batch_size + 1}: {len(batch)} segs in {batch_latency:.0f} ms")

        total_ms = (time.perf_counter() - total_start) * 1000
        return BatchTranslationResult(
            results=all_results,
            total_latency_ms=round(total_ms, 2),
            avg_latency_ms=round(total_ms / len(texts), 2) if texts else 0.0,
        )

    def gpu_info(self) -> dict:
        if not torch.cuda.is_available():
            return {"device": "cpu"}
        return {
            "device":       torch.cuda.get_device_name(0),
            "allocated_gb": round(torch.cuda.memory_allocated(0) / 1e9, 2),
            "reserved_gb":  round(torch.cuda.memory_reserved(0)  / 1e9, 2),
            "total_gb":     round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2),
        }

### Load Models

In [5]:
translator = MarianTranslator()
print(translator.gpu_info())

INFO | Loading ES→EN model: Helsinki-NLP/opus-mt-es-en
INFO | HTTP Request: HEAD https://huggingface.co/Helsinki-NLP/opus-mt-es-en/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Helsinki-NLP/opus-mt-es-en/c96e2c5399ebfae4fc43d9669556b9afa74bb69d/tokenizer_config.json "HTTP/1.1 200 OK"
INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Helsinki-NLP/opus-mt-es-en/c96e2c5399ebfae4fc43d9669556b9afa74bb69d/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

INFO | HTTP Request: GET https://huggingface.co/api/models/Helsinki-NLP/opus-mt-es-en/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO | HTTP Request: GET https://huggingface.co/api/models/Helsinki-NLP/opus-mt-es-en/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO | HTTP Request: HEAD https://huggingface.co/Helsinki-NLP/opus-mt-es-en/resolve/main/source.spm "HTTP/1.1 307 Temporary Redirect"
INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Helsinki-NLP/opus-mt-es-en/c96e2c5399ebfae4fc43d9669556b9afa74bb69d/source.spm "HTTP/1.1 200 OK"
INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Helsinki-NLP/opus-mt-es-en/c96e2c5399ebfae4fc43d9669556b9afa74bb69d/source.spm "HTTP/1.1 200 OK"


source.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

INFO | HTTP Request: HEAD https://huggingface.co/Helsinki-NLP/opus-mt-es-en/resolve/main/target.spm "HTTP/1.1 307 Temporary Redirect"
INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Helsinki-NLP/opus-mt-es-en/c96e2c5399ebfae4fc43d9669556b9afa74bb69d/target.spm "HTTP/1.1 200 OK"
INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Helsinki-NLP/opus-mt-es-en/c96e2c5399ebfae4fc43d9669556b9afa74bb69d/target.spm "HTTP/1.1 200 OK"


...

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

INFO | HTTP Request: GET https://huggingface.co/api/models/Helsinki-NLP/opus-mt-en-es/xet-read-token/fdaddf76f50fcc1583ba42f95965862a7ab30f97 "HTTP/1.1 200 OK"


model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

INFO | Both models ready


{'device': 'Tesla T4', 'allocated_gb': 1.16, 'reserved_gb': 1.17, 'total_gb': 15.64}


### Smoke Test

In [6]:
# ES → EN
text_es = "El paciente tiene fiebre y dolor de cabeza."
result_en = translator.translate(text_es, src="es", tgt="en")
print(f"Spanish : {text_es}")
print(f"English : {result_en.translated_text}")
print(f"Latency : {result_en.latency_ms} ms\n")

# EN → ES
text_en = "The engine requires an oil change every 5,000 miles."
result_es = translator.translate(text_en, src="en", tgt="es")
print(f"English : {text_en}")
print(f"Spanish : {result_es.translated_text}")
print(f"Latency : {result_es.latency_ms} ms")

Spanish : El paciente tiene fiebre y dolor de cabeza.
English : The patient has a fever and headache.
Latency : 2343.4 ms

English : The engine requires an oil change every 5,000 miles.
Spanish : El motor requiere un cambio de aceite cada 5.000 millas.
Latency : 105.84 ms


---
## Baseline Benchmark — BLEU

Same test cases as the NLLB-200 baseline for direct comparison. Added a Legal example to broaden domain coverage.

In [7]:
!pip install sacrebleu

  Using cached sacrebleu-2.6.0-py3-none-any.whl.metadata (39 kB)
  Using cached portalocker-3.2.0-py3-none-any.whl.metadata (8.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 1.0 MB/s eta 0:00:00 0:00:01


In [8]:
import pandas as pd
import sacrebleu

# (source_text, src_lang, tgt_lang, domain, reference)
benchmark_cases = [
    ("El cielo es azul y el sol brilla.",                        "es", "en", "General",    "The sky is blue and the sun shines."),
    ("The quick brown fox jumps over the lazy dog.",             "en", "es", "General",    "El rápido zorro marrón salta sobre el perro perezoso."),
    ("El paciente presenta fiebre alta y tos seca.",             "es", "en", "Medical",    "The patient presents with high fever and a dry cough."),
    ("The torque converter must be replaced.",                   "en", "es", "Automotive", "El convertidor de par debe ser reemplazado."),
    ("El contrato incluye una cláusula de confidencialidad.",    "es", "en", "Legal",      "The contract includes a confidentiality clause."),
    ("The parties agree to resolve disputes by arbitration.",    "en", "es", "Legal",      "Las partes acuerdan resolver disputas mediante arbitraje."),
]

bleu_rows = []

print(f"{'Domain':<12} {'Dir':<8} {'BLEU':>7} {'Latency':>10}  Source / Translation")
print("─" * 95)

for text, src, tgt, domain, reference in benchmark_cases:
    r = translator.translate(text, src=src, tgt=tgt)
    bleu = sacrebleu.sentence_bleu(r.translated_text, [reference]).score
    direction = f"{src.upper()}→{tgt.upper()}"

    print(f"[{domain:<10}] {direction:<8} {bleu:>7.2f} {r.latency_ms:>8.0f} ms")
    print(f"  SRC: {r.source_text}")
    print(f"  TGT: {r.translated_text}")
    print(f"  REF: {reference}\n")

    bleu_rows.append({
        "Domain":      domain,
        "Direction":   direction,
        "Source":      r.source_text,
        "Translation": r.translated_text,
        "Reference":   reference,
        "BLEU":        round(bleu, 2),
        "Latency_ms":  r.latency_ms,
        "Model":       r.model,
    })

Domain       Dir         BLEU    Latency  Source / Translation
───────────────────────────────────────────────────────────────────────────────────────────────
[General   ] ES→EN      66.06      104 ms
  SRC: El cielo es azul y el sol brilla.
  TGT: The sky is blue and the sun is shining.
  REF: The sky is blue and the sun shines.

[General   ] EN→ES      79.56      122 ms
  SRC: The quick brown fox jumps over the lazy dog.
  TGT: El zorro marrón salta sobre el perro perezoso.
  REF: El rápido zorro marrón salta sobre el perro perezoso.

[Medical   ] ES→EN      63.16       95 ms
  SRC: El paciente presenta fiebre alta y tos seca.
  TGT: The patient has a high fever and a dry cough.
  REF: The patient presents with high fever and a dry cough.

[Automotive] EN→ES      22.77       85 ms
  SRC: The torque converter must be replaced.
  TGT: Debe sustituirse el convertidor de par.
  REF: El convertidor de par debe ser reemplazado.

[Legal     ] ES→EN     100.00       72 ms
  SRC: El contrato 

---
## Full Evaluation — COMET + BERTScore

Same metric stack used in the domain-specific fine-tuning pipeline (`sp-test.ipynb`), so baseline and post-fine-tuning scores are directly comparable.

In [11]:
from comet import download_model, load_from_checkpoint

comet_model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(comet_model_path)
print("COMET model loaded")

ModuleNotFoundError: No module named 'comet'

In [ ]:
from bert_score import score as bert_score_fn

# Pull translations already computed in BLEU cell
sources      = [r["Source"]      for r in bleu_rows]
translations = [r["Translation"] for r in bleu_rows]
references   = [r["Reference"]   for r in bleu_rows]
directions   = [r["Direction"]   for r in bleu_rows]

# COMET (reference-based)
comet_data = [
    {"src": s, "mt": t, "ref": rf}
    for s, t, rf in zip(sources, translations, references)
]
comet_output = comet_model.predict(
    comet_data, batch_size=8,
    gpus=1 if torch.cuda.is_available() else 0
)
comet_scores = comet_output.scores

# BERTScore — evaluate EN outputs against EN references,
# ES outputs against ES references
en_indices = [i for i, d in enumerate(directions) if d.endswith("EN")]
es_indices = [i for i, d in enumerate(directions) if d.endswith("ES")]

bert_f1 = [0.0] * len(translations)

if en_indices:
    _, _, f1_en = bert_score_fn(
        [translations[i] for i in en_indices],
        [references[i]   for i in en_indices],
        lang="en",
        model_type="microsoft/deberta-xlarge-mnli",
        verbose=False,
    )
    for idx, i in enumerate(en_indices):
        bert_f1[i] = f1_en[idx].item()

if es_indices:
    _, _, f1_es = bert_score_fn(
        [translations[i] for i in es_indices],
        [references[i]   for i in es_indices],
        lang="es",
        verbose=False,
    )
    for idx, i in enumerate(es_indices):
        bert_f1[i] = f1_es[idx].item()

print("COMET and BERTScore computed")

In [ ]:
# Thresholds (same as sp-test.ipynb)
COMET_AUTO        = 0.88
COMET_REPAIR_MIN  = 0.75
BERTSCORE_MIN     = 0.85

GREEN  = "\033[92m"
YELLOW = "\033[93m"
RED    = "\033[91m"
RESET  = "\033[0m"

def classify(comet, bert):
    if comet >= COMET_AUTO and bert >= BERTSCORE_MIN:
        return "auto-accept", GREEN
    elif comet >= COMET_REPAIR_MIN:
        return "auto-repair", YELLOW
    else:
        return "human-review", RED

for i, row in enumerate(bleu_rows):
    cs = comet_scores[i]
    bs = bert_f1[i]
    action, colour = classify(cs, bs)

    print(f"{'='*65}")
    print(f"[{row['Domain']:<10}] {row['Direction']}")
    print(f"Source:      {row['Source']}")
    print(f"Translation: {colour}{row['Translation']}{RESET}")
    print(f"Reference:   {row['Reference']}")
    print(f"Metrics:     BLEU={row['BLEU']:.2f}  COMET={cs:.3f}  BERTScore={bs:.3f}  → {colour}{action}{RESET}")
    print()

    # Attach rich metrics back to row
    row["COMET"]          = round(cs, 4)
    row["BERTScore"]      = round(bs, 4)
    row["Classification"] = action

---
## Latency Benchmarking (Batch)

In [ ]:
batch_segments = [
    "El contrato incluye una cláusula de confidencialidad.",
    "El vehículo debe pasar una inspección técnica anual.",
    "Se recomienda reposo absoluto durante 48 horas.",
    "La sentencia fue apelada ante el tribunal superior.",
]

batch_result = translator.translate_batch(batch_segments, src="es", tgt="en", batch_size=4)

print(f"Total  : {batch_result.total_latency_ms:.0f} ms")
print(f"Avg    : {batch_result.avg_latency_ms:.0f} ms / segment\n")

for r in batch_result.results:
    print(f"  ES: {r.source_text}")
    print(f"  EN: {r.translated_text}\n")

---
## Export Baseline Report

In [ ]:
import os

os.makedirs("results", exist_ok=True)

df_baseline = pd.DataFrame(bleu_rows)

# Reorder columns
col_order = ["Domain", "Direction", "Source", "Translation", "Reference",
             "BLEU", "COMET", "BERTScore", "Classification", "Latency_ms", "Model"]
df_baseline = df_baseline[col_order]

csv_path = "results/MarianMT_Baseline_Report.csv"
df_baseline.to_csv(csv_path, index=False, encoding="utf-8-sig")

print(f"Baseline report saved to: {csv_path}")
print()
print(df_baseline[["Domain", "Direction", "BLEU", "COMET", "BERTScore", "Classification"]].to_string(index=False))

---
## Next Steps
1. **General Base Model** (this notebook) ✅
2. **Domain fine-tuning** — Medical (`sp-test.ipynb`), then Legal, Automotive
3. **QA Module** — compare fine-tuned COMET/BERTScore against this baseline
4. **Automated Repair** — Mistral-7B + T5 for tag repair on auto-repair segments
5. **Streamlit / Gradio prototype** with confidence heatmap UI